In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

############################################
# 0. 配置区：把这些按你本地实际情况改一下
############################################
train_path = "ruc_Class25Q2_train_price_clean3.csv"   # 你的“买房训练集”（有Price）
test_path  = "副本ruc_Class25Q2_test_price_final.csv"    # 你的“买房测试集”（要预测Price的这批）

TARGET_COL = "Price"
BOARD_COL  = "板块"     # 小区/片区列名
TIME_COL   = "time_index"

# KMeans 聚类簇数：买房用 K=30
K_CLUSTERS = 30

# 这三组正则超参请用“买房数据”那套调参结果
RIDGE_ALPHA_BEST   = 10000.0      # Ridge最优alpha（示例）
LASSO_ALPHA_BEST   = 100.0    # Lasso最优alpha（示例）
ENET_ALPHA_BEST    = 1.0      # 弹性网最佳alpha（示例）
ENET_L1RATIO_BEST  = 0.8        # 弹性网最佳l1_ratio（示例）

# 导出excel文件名
OUT_OLS_XLS        = "preds_buy_OLS.xlsx"
OUT_RIDGE_XLS      = "preds_buy_Ridge.xlsx"
OUT_LASSO_XLS      = "preds_buy_Lasso.xlsx"
OUT_ENET_XLS       = "preds_buy_ElasticNet.xlsx"

############################################
# 1. 读训练 / 测试
############################################
df_tr_raw = pd.read_csv(train_path)
df_te_raw = pd.read_csv(test_path)

# 备份测试集原始表，后面拼预测列然后导出
df_te_export_base = df_te_raw.copy()

############################################
# 2. 把训练+测试先拼一起做“板块画像”然后再拆
#    （但KMeans只用训练集去fit）
############################################

def to_numeric_inplace(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

# 2.1 先把我们会用到的楼盘/地段特征转成数值
block_profile_cols = [
    "lon","lat",
    TARGET_COL,        # 平均价，只有训练集有，测试集可能全是NaN也没事
    "房龄",
    "绿 化 率","容 积 率","物 业 费",
    "房屋总数","楼栋总数",
    "subway"
]
for col in block_profile_cols:
    to_numeric_inplace(df_tr_raw, [col])
    to_numeric_inplace(df_te_raw, [col])

# 2.2 训练集按“板块”算板块画像
#     注意：这里用训练集，因为测试集没有Price（没法给出真实均价）
temp_tr = df_tr_raw.copy()
temp_tr["有电梯_01"] = 1 - temp_tr["配备电梯_无_01"] if "配备电梯_无_01" in temp_tr.columns else np.nan
to_numeric_inplace(temp_tr, ["有电梯_01"])

block_profile = (
    temp_tr.groupby(BOARD_COL)
    .agg(
        n_obs            = (BOARD_COL,"size"),
        center_lon       = ("lon","mean"),
        center_lat       = ("lat","mean"),
        avg_price        = (TARGET_COL,"mean"),
        avg_age          = ("房龄","mean"),
        avg_green        = ("绿 化 率","mean"),
        avg_plot_ratio   = ("容 积 率","mean"),
        avg_property_fee = ("物 业 费","mean"),
        avg_subway       = ("subway","mean"),
        share_elevator   = ("有电梯_01","mean"),
    )
    .reset_index()
)

# 缺失填中位数，保证KMeans能跑
fill_cols_for_cluster = [
    "center_lon","center_lat","avg_price","avg_age",
    "avg_green","avg_plot_ratio","avg_property_fee",
    "avg_subway","share_elevator"
]
for c in fill_cols_for_cluster:
    block_profile[c] = pd.to_numeric(block_profile[c], errors="coerce")
    block_profile[c] = block_profile[c].fillna(block_profile[c].median())

############################################
# 3. 对板块画像做KMeans (K=30)，得到 market_cluster30
############################################
cluster_features = [
    "center_lon","center_lat",
    "avg_price","avg_age",
    "avg_green","avg_plot_ratio",
    "avg_property_fee","avg_subway","share_elevator"
]
X_block = block_profile[cluster_features].values

scaler_block = StandardScaler()
X_block_scaled = scaler_block.fit_transform(X_block)

K_use = min(K_CLUSTERS, len(block_profile))
kmeans_block = KMeans(n_clusters=K_use, random_state=42, n_init=10)
block_profile["market_cluster30"] = kmeans_block.fit_predict(X_block_scaled).astype(int)

############################################
# 4. 把 cluster30 贴回训练集和测试集
############################################
df_tr = df_tr_raw.merge(
    block_profile[[BOARD_COL,"market_cluster30"]],
    on=BOARD_COL,
    how="left"
)
df_te = df_te_raw.merge(
    block_profile[[BOARD_COL,"market_cluster30"]],
    on=BOARD_COL,
    how="left"
)

############################################
# 5. 加交互项（可选但推荐，和之前线性思路一致）
############################################
def add_interactions(df):
    out = df.copy()

    # 面积 × 房龄
    if "建筑面积" in out.columns and "房龄" in out.columns:
        out["交互_面积x房龄"] = pd.to_numeric(out["建筑面积"], errors="coerce") * \
                           pd.to_numeric(out["房龄"], errors="coerce")

    # 面积 × 高楼层
    if "建筑面积" in out.columns and "高楼层_01" in out.columns:
        out["交互_面积x高楼层"] = pd.to_numeric(out["建筑面积"], errors="coerce") * \
                             pd.to_numeric(out["高楼层_01"], errors="coerce")

    # 高楼层 × 有电梯
    if "高楼层_01" in out.columns and "配备电梯_无_01" in out.columns:
        # 有电梯 = 1 - 无电梯
        tmp_ele = 1 - pd.to_numeric(out["配备电梯_无_01"], errors="coerce")
        out["交互_高楼层x有电梯"] = pd.to_numeric(out["高楼层_01"], errors="coerce") * tmp_ele

    return out

df_tr = add_interactions(df_tr)
df_te = add_interactions(df_te)

############################################
# 6. 组特征矩阵 X_train / X_test
############################################
# city_xx dummy 已经是列了（city_00, city_01 ...）你之前数据就是一堆0/1列
# 楼层、朝向、装修 这些也已经是 0/1 列
# 我们只要把它们全拿进来

base_feature_cols = [
    "建筑面积",
    "lon","lat",
    "房屋总数","楼栋总数","绿 化 率","容 积 率","物 业 费",
    "subway",
    "室","厅","厨","卫",
    "地下室_01","底层_01","低楼层_01","高楼层_01","顶层_01",
    "配备电梯_无_01",
    "朝向_东_01","朝向_南_01","朝向_西_01","朝向_北_01",
    "装修_简装_01","装修_毛坯_01","装修_其他_01",
    "房龄",
    TIME_COL,
    # 城市dummy（city_00 ... city_11 之类）
    # 我们会自动找所有以 "city_" 开头的列
    # cluster30 dummy 稍后加
    "交互_面积x房龄","交互_面积x高楼层","交互_高楼层x有电梯",
]

def build_design_matrix(df):
    df2 = df.copy()

    # 先把已有列转成数值
    for c in df2.columns:
        if c in base_feature_cols or c.startswith("city_") or c == "market_cluster30":
            df2[c] = pd.to_numeric(df2[c], errors="coerce")

    # cluster30 -> one-hot
    cluster_dum = pd.get_dummies(
        df2["market_cluster30"],
        prefix="cluster30",
        drop_first=False
    ).astype(int)

    # 城市dummy：直接抓以 city_ 开头的列
    city_cols = [c for c in df2.columns if c.startswith("city_")]
    city_df = df2[city_cols].copy() if city_cols else pd.DataFrame(index=df2.index)

    # 主特征
    kept_main_cols = [c for c in base_feature_cols if c in df2.columns]
    X_main = df2[kept_main_cols].copy()

    # 数值化+缺失填
    for c in X_main.columns:
        X_main[c] = pd.to_numeric(X_main[c], errors="coerce")
    X_main = X_main.fillna(X_main.median(numeric_only=True))

    # 合并
    X_full_local = pd.concat(
        [X_main.reset_index(drop=True),
         city_df.reset_index(drop=True),
         cluster_dum.reset_index(drop=True)],
        axis=1
    )

    # 去掉标准差为0的列（全常数列）
    X_full_local = X_full_local.loc[:, X_full_local.std(axis=0) > 0]

    return X_full_local

X_tr_full = build_design_matrix(df_tr)
X_te_full = build_design_matrix(df_te)

# 重要：保证测试集列顺序和训练集完全一致
train_cols = X_tr_full.columns
X_te_full = X_te_full.reindex(columns=train_cols, fill_value=0)

############################################
# 7. 训练目标 y + IQR 去极值（只对训练）
############################################
y_tr_raw = pd.to_numeric(df_tr[TARGET_COL], errors="coerce")
y_tr_raw = y_tr_raw.fillna(y_tr_raw.median())

Q1 = y_tr_raw.quantile(0.25)
Q3 = y_tr_raw.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_tr_raw >= lower_cut) & (y_tr_raw <= upper_cut)

X_ok = X_tr_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_tr_raw.loc[mask_ok].reset_index(drop=True)

# 测试集保持原样（不滤）
X_test_final = X_te_full.reset_index(drop=True)

print("训练样本数(滤极值后):", X_ok.shape)
print("测试样本数:", X_test_final.shape)

############################################
# 8. 定义四个模型
############################################

# OLS
model_ols = make_pipeline(
    StandardScaler(with_mean=False),
    LinearRegression()
)

# Ridge
model_ridge = make_pipeline(
    StandardScaler(with_mean=False),
    Ridge(
        alpha=RIDGE_ALPHA_BEST,
        fit_intercept=True,
        random_state=42
    )
)

# Lasso
model_lasso = make_pipeline(
    StandardScaler(with_mean=False),
    Lasso(
        alpha=LASSO_ALPHA_BEST,
        fit_intercept=True,
        max_iter=50000,
        random_state=42
    )
)

# 弹性网络
model_enet = make_pipeline(
    StandardScaler(with_mean=False),
    ElasticNet(
        alpha=ENET_ALPHA_BEST,
        l1_ratio=ENET_L1RATIO_BEST,
        fit_intercept=True,
        max_iter=50000,
        random_state=42
    )
)

############################################
# 9. 拟合四个模型，用训练集(去极值后)
############################################
print("拟合 OLS...")
model_ols.fit(X_ok, y_ok)

print("拟合 Ridge...")
model_ridge.fit(X_ok, y_ok)

print("拟合 Lasso...")
model_lasso.fit(X_ok, y_ok)

print("拟合 ElasticNet...")
model_enet.fit(X_ok, y_ok)

############################################
# 10. 预测测试集
############################################
pred_ols   = model_ols.predict(X_test_final)
pred_ridge = model_ridge.predict(X_test_final)
pred_lasso = model_lasso.predict(X_test_final)
pred_enet  = model_enet.predict(X_test_final)

############################################
# 11. 拼回测试集原始行并导出Excel
############################################
te_out_ols   = df_te_export_base.copy()
te_out_ridge = df_te_export_base.copy()
te_out_lasso = df_te_export_base.copy()
te_out_enet  = df_te_export_base.copy()

te_out_ols["Pred_OLS"]                 = pred_ols
te_out_ridge["Pred_Ridge"]             = pred_ridge
te_out_lasso["Pred_Lasso"]             = pred_lasso
te_out_enet["Pred_ElasticNet"]         = pred_enet

# 导出
te_out_ols.to_excel(OUT_OLS_XLS, index=False)
te_out_ridge.to_excel(OUT_RIDGE_XLS, index=False)
te_out_lasso.to_excel(OUT_LASSO_XLS, index=False)
te_out_enet.to_excel(OUT_ENET_XLS, index=False)

print("导出完成：")
print(" ", OUT_OLS_XLS)
print(" ", OUT_RIDGE_XLS)
print(" ", OUT_LASSO_XLS)
print(" ", OUT_ENET_XLS)


KeyError: "Column(s) ['subway'] do not exist"

In [5]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

############################################
# 0. 配置区：改成你的真实文件路径
############################################
train_path = "副本ruc_Class25Q2_train_price_clean3.csv"   # 训练集（含Price）
test_path  = "要用的数据.csv"    # 测试集（不含Price，要预测）

TARGET_COL = "Price"

# 你想分成多少个市场簇，这里是买房版：30个
K_CLUSTERS = 30

# 四个模型的超参（用你针对“买房数据”调出来的）
RIDGE_ALPHA_BEST   = 10000.0       # Ridge最优alpha
LASSO_ALPHA_BEST   = 100.0     # Lasso最优alpha
ENET_ALPHA_BEST    = 1.0       # 弹性网alpha
ENET_L1RATIO_BEST  = 0.8       # 弹性网l1_ratio (0~1之间)

# 输出文件名
OUT_OLS_XLS        = "preds_buy_OLS.xlsx"
OUT_RIDGE_XLS      = "preds_buy_Ridge.xlsx"
OUT_LASSO_XLS      = "preds_buy_Lasso.xlsx"
OUT_ENET_XLS       = "preds_buy_ElasticNet.xlsx"

############################################
# 1. 读取训练 / 测试
############################################
df_tr_raw = pd.read_csv(train_path)
df_te_raw = pd.read_csv(test_path)

# 备份测试集原始行，最后拼预测列
df_te_export = df_te_raw.copy()

############################################
# 2. 清洗成数值 & 做交互项
############################################
def to_numeric_cols(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

def add_interactions(df):
    out = df.copy()
    # 面积 × 房龄
    if "建筑面积" in out.columns and "房龄" in out.columns:
        out["交互_面积x房龄"] = (
            pd.to_numeric(out["建筑面积"], errors="coerce")
            * pd.to_numeric(out["房龄"], errors="coerce")
        )
    # 面积 × 高楼层
    if "建筑面积" in out.columns and "高楼层_01" in out.columns:
        out["交互_面积x高楼层"] = (
            pd.to_numeric(out["建筑面积"], errors="coerce")
            * pd.to_numeric(out["高楼层_01"], errors="coerce")
        )
    # 高楼层 × 有电梯(=1-无电梯)
    if "高楼层_01" in out.columns and "配备电梯_无_01" in out.columns:
        has_elevator = 1 - pd.to_numeric(out["配备电梯_无_01"], errors="coerce")
        out["交互_高楼层x有电梯"] = (
            pd.to_numeric(out["高楼层_01"], errors="coerce") * has_elevator
        )
    return out

# 我们先把训练集/测试集做数值化 + 交互项
numeric_like_cols = [
    TARGET_COL,
    "建筑面积","房龄","室","厅","厨","卫",
    "房屋总数","楼栋总数","绿 化 率","容 积 率","物 业 费",
    "subway",
    "lon","lat",
    "地下室_01","底层_01","低楼层_01","高楼层_01","顶层_01",
    "配备电梯_无_01",
    "朝向_东_01","朝向_南_01","朝向_西_01","朝向_北_01",
    "建筑结构_混合结构_01","建筑结构_砖混结构_01","建筑结构_未知结构_01",
    "建筑结构_框架结构_01","建筑结构_钢结构_01",
    "装修_简装_01","装修_其他_01","装修_毛坯_01",
    # 城市dummy:
    "city_00","city_01","city_03","city_04","city_05","city_06",
    "city_07","city_08","city_09","city_10","city_11",
    # 年份dummy:
    "year_2019","year_2020","year_2021","year_2022","year_2023","year_2024","year_2025",
    # 月份dummy:
    "month_2","month_3","month_4","month_5","month_6","month_7",
    "month_8","month_9","month_10","month_11","month_12"
]

# 转数值
to_numeric_cols(df_tr_raw, numeric_like_cols)
to_numeric_cols(df_te_raw, numeric_like_cols)

# 交互项
df_tr = add_interactions(df_tr_raw)
df_te = add_interactions(df_te_raw)

############################################
# 3. 用训练集给每一套房打“市场分区 cluster30”
#    - 用位置/小区质量特征直接对房源聚类，不再依赖“板块”列
############################################
cluster_basis_cols = [
    "lon","lat",
    "房龄","绿 化 率","容 积 率","物 业 费",
    "房屋总数","楼栋总数",
    "subway"
]
# 训练时的聚类数据
train_cluster_df = df_tr[cluster_basis_cols].copy()

# 缺失值补中位数，防止KMeans崩
for c in train_cluster_df.columns:
    train_cluster_df[c] = pd.to_numeric(train_cluster_df[c], errors="coerce")
train_cluster_df = train_cluster_df.fillna(train_cluster_df.median(numeric_only=True))

# 训练标准化+KMeans
scaler_cluster = StandardScaler()
train_scaled = scaler_cluster.fit_transform(train_cluster_df)

k_use = min(K_CLUSTERS, len(df_tr))
kmeans_model = KMeans(n_clusters=k_use, random_state=42, n_init=10)
df_tr["market_cluster30"] = kmeans_model.fit_predict(train_scaled).astype(int)

# 测试集同样流程：同样列 -> 填训练集的中位数 -> transform -> predict
test_cluster_df = df_te[cluster_basis_cols].copy()
for c in test_cluster_df.columns:
    test_cluster_df[c] = pd.to_numeric(test_cluster_df[c], errors="coerce")
test_cluster_df = test_cluster_df.fillna(train_cluster_df.median(numeric_only=True))

test_scaled = scaler_cluster.transform(test_cluster_df)
df_te["market_cluster30"] = kmeans_model.predict(test_scaled).astype(int)

############################################
# 4. 组最终特征矩阵 X_train / X_test
############################################
# 我们要的一次性所有解释变量：
#   - 结构/户型/面积/楼层/电梯/朝向/装修/房龄
#   - 小区特征(绿化率/容积率/物业费/房屋总数/楼栋总数/subway)
#   - 经纬度
#   - 城市dummy
#   - 年份/月份dummy
#   - 交互项 (面积x房龄, 面积x高楼层, 高楼层x电梯)
#   - market_cluster30 的 dummy

base_feature_cols = [
    "建筑面积","房龄","室","厅","厨","卫",
    "房屋总数","楼栋总数","绿 化 率","容 积 率","物 业 费",
    "subway",
    "lon","lat",

    "地下室_01","底层_01","低楼层_01","高楼层_01","顶层_01",
    "配备电梯_无_01",

    "朝向_东_01","朝向_南_01","朝向_西_01","朝向_北_01",

    "建筑结构_混合结构_01","建筑结构_砖混结构_01","建筑结构_未知结构_01",
    "建筑结构_框架结构_01","建筑结构_钢结构_01",

    "装修_简装_01","装修_其他_01","装修_毛坯_01",

    # 城市dummy:
    "city_00","city_01","city_03","city_04","city_05","city_06",
    "city_07","city_08","city_09","city_10","city_11",

    # 年份dummy:
    "year_2019","year_2020","year_2021","year_2022","year_2023","year_2024","year_2025",

    # 月份dummy:
    "month_2","month_3","month_4","month_5","month_6","month_7",
    "month_8","month_9","month_10","month_11","month_12",

    # 交互项:
    "交互_面积x房龄","交互_面积x高楼层","交互_高楼层x有电梯",
]

def build_design_matrix(df):
    df2 = df.copy()

    # cluster30 → dummy
    cluster_dum = pd.get_dummies(
        df2["market_cluster30"],
        prefix="cluster30",
        drop_first=False
    ).astype(int)

    # 主特征存在才取
    use_cols = [c for c in base_feature_cols if c in df2.columns]
    X_main = df2[use_cols].copy()

    # 数值化
    for c in X_main.columns:
        X_main[c] = pd.to_numeric(X_main[c], errors="coerce")
    # 缺失填中位数
    X_main = X_main.fillna(X_main.median(numeric_only=True))

    # 合并主特征+cluster dummy
    X_full_local = pd.concat(
        [X_main.reset_index(drop=True),
         cluster_dum.reset_index(drop=True)],
        axis=1
    )

    # 去掉标准差=0的列（常数列）
    X_full_local = X_full_local.loc[:, X_full_local.std(axis=0) > 0]

    return X_full_local

X_tr_full = build_design_matrix(df_tr)
X_te_full = build_design_matrix(df_te)

# 统一列顺序：测试集按训练集列来排，缺的补0
train_cols = X_tr_full.columns
X_te_full = X_te_full.reindex(columns=train_cols, fill_value=0)

############################################
# 5. 目标y + IQR滤极端值（只滤训练）
############################################
y_tr_raw = pd.to_numeric(df_tr[TARGET_COL], errors="coerce")
y_tr_raw = y_tr_raw.fillna(y_tr_raw.median())

Q1 = y_tr_raw.quantile(0.25)
Q3 = y_tr_raw.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_tr_raw >= lower_cut) & (y_tr_raw <= upper_cut)

X_ok = X_tr_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_tr_raw.loc[mask_ok].reset_index(drop=True)

X_test_final = X_te_full.reset_index(drop=True)

print("训练样本(过滤后):", X_ok.shape)
print("测试样本:", X_test_final.shape)

############################################
# 6. 定义四个模型：OLS / Ridge / Lasso / 弹性网
############################################
from sklearn.preprocessing import StandardScaler

# 注意：我们和前面保持一致，用 StandardScaler(with_mean=False)
# 这样不会因为大量0/1列导致报错（稀疏/虚拟变量）

model_ols = make_pipeline(
    StandardScaler(with_mean=False),
    LinearRegression()
)

model_ridge = make_pipeline(
    StandardScaler(with_mean=False),
    Ridge(
        alpha=RIDGE_ALPHA_BEST,
        fit_intercept=True,
        random_state=42
    )
)

model_lasso = make_pipeline(
    StandardScaler(with_mean=False),
    Lasso(
        alpha=LASSO_ALPHA_BEST,
        fit_intercept=True,
        max_iter=50000,
        random_state=42
    )
)

model_enet = make_pipeline(
    StandardScaler(with_mean=False),
    ElasticNet(
        alpha=ENET_ALPHA_BEST,
        l1_ratio=ENET_L1RATIO_BEST,
        fit_intercept=True,
        max_iter=50000,
        random_state=42
    )
)

############################################
# 7. 拟合四个模型
############################################
print("拟合 OLS ...")
model_ols.fit(X_ok, y_ok)

print("拟合 Ridge ...")
model_ridge.fit(X_ok, y_ok)

print("拟合 Lasso ...")
model_lasso.fit(X_ok, y_ok)

print("拟合 ElasticNet ...")
model_enet.fit(X_ok, y_ok)

############################################
# 8. 预测测试集
############################################
pred_ols   = model_ols.predict(X_test_final)
pred_ridge = model_ridge.predict(X_test_final)
pred_lasso = model_lasso.predict(X_test_final)
pred_enet  = model_enet.predict(X_test_final)

############################################
# 9. 拼回测试集原始数据并导出4个Excel
############################################
out_ols   = df_te_export.copy()
out_ridge = df_te_export.copy()
out_lasso = df_te_export.copy()
out_enet  = df_te_export.copy()

out_ols["Pred_OLS"]             = pred_ols
out_ridge["Pred_Ridge"]         = pred_ridge
out_lasso["Pred_Lasso"]         = pred_lasso
out_enet["Pred_ElasticNet"]     = pred_enet

out_ols.to_excel(OUT_OLS_XLS, index=False)
out_ridge.to_excel(OUT_RIDGE_XLS, index=False)
out_lasso.to_excel(OUT_LASSO_XLS, index=False)
out_enet.to_excel(OUT_ENET_XLS, index=False)

print("导出完成：")
print(" ", OUT_OLS_XLS)
print(" ", OUT_RIDGE_XLS)
print(" ", OUT_LASSO_XLS)
print(" ", OUT_ENET_XLS)


训练样本(过滤后): (96048, 76)
测试样本: (34017, 76)
拟合 OLS ...
拟合 Ridge ...
拟合 Lasso ...
拟合 ElasticNet ...
导出完成：
  preds_buy_OLS.xlsx
  preds_buy_Ridge.xlsx
  preds_buy_Lasso.xlsx
  preds_buy_ElasticNet.xlsx


In [6]:
# 假设我们已经有：
# df_te_export   -> 原始测试集（包含 ID 列）
# pred_ols       -> OLS 预测出来的价格
# pred_ridge     -> Ridge 预测出来的价格
# pred_lasso     -> Lasso 预测出来的价格
# pred_enet      -> ElasticNet 预测出来的价格

# 1. 先确保 ID 是在的
if "ID" not in df_te_export.columns:
    raise ValueError("测试集里没有 ID 列，没法按你要的格式导出。请确认测试集包含 ID。")

# 2. 针对四个模型分别做两列的小表
sub_ols = pd.DataFrame({
    "ID": df_te_export["ID"].values,
    "Price": pred_ols
})

sub_ridge = pd.DataFrame({
    "ID": df_te_export["ID"].values,
    "Price": pred_ridge
})

sub_lasso = pd.DataFrame({
    "ID": df_te_export["ID"].values,
    "Price": pred_lasso
})

sub_enet = pd.DataFrame({
    "ID": df_te_export["ID"].values,
    "Price": pred_enet
})

# 3. 导出成 Excel（或者CSV，如果老师/比赛要求csv）
sub_ols.to_excel("preds_buy_OLS.xlsx", index=False)
sub_ridge.to_excel("preds_buy_Ridge.xlsx", index=False)
sub_lasso.to_excel("preds_buy_Lasso.xlsx", index=False)
sub_enet.to_excel("preds_buy_ElasticNet.xlsx", index=False)

# 也可以同时吐CSV版本（看你要不要）
sub_ols.to_csv("preds_buy_OLS.csv", index=False)
sub_ridge.to_csv("preds_buy_Ridge.csv", index=False)
sub_lasso.to_csv("preds_buy_Lasso.csv", index=False)
sub_enet.to_csv("preds_buy_ElasticNet.csv", index=False)

print("导出完成（两列格式：ID, Price）")


导出完成（两列格式：ID, Price）
